# DATA PROCESSING

## Initialisation spark et chargement data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, schema_of_json
from pyspark.sql.types import StringType, IntegerType, DoubleType, ArrayType, StructType
import os
from pathlib import Path

#project_root = Path(__file__).parent.parent
app_name="ArXivArticle-Classification"
master="spark://tawfekh-d:7077"#"spark://$(hostname):7077"  #"local[*]"   # Remplacer par cluster URL en production
memory= "4g"
executor_memory= "2g"
driver_memory= "2g"
# cores_per_executor: 4
# num_executors: 2
  
def _create_spark_session():
        """Créer une session Spark optimisée"""
        return SparkSession.builder \
            .appName(app_name) \
            .getOrCreate()
            # .master(master) \
            # .config('spark.driver.memory', memory) \
            # .config('spark.driver.executor.memory', executor_memory) \
            # .config('spark.sql.adaptive.enabled', 'true') \
            # .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
            
spark=_create_spark_session()

In [ ]:
def clean_text_column(self, df, text_col, new_col=None):
        """
        Nettoie une colonne de texte
        """
        if new_col is None:
            new_col = f"clean_{text_col}"
        
        print(f" Nettoyage de la colonne: {text_col}")
        
        # Chaîne de nettoyage
        cleaned = F.lower(F.col(text_col))  # Minuscules
        
        # Supprimer les caractères spéciaux (garder lettres, chiffres, espaces)
        cleaned = F.regexp_replace(cleaned, r'[^a-zA-Z0-9\s]', ' ')
        
        # Supprimer les URLs
        cleaned = F.regexp_replace(cleaned, r'https?://\S+|www\.\S+', ' ')
        
        # Supprimer les nombres isolés
        cleaned = F.regexp_replace(cleaned, r'\b\d+\b', ' ')
        
        # Supprimer les espaces multiples
        cleaned = F.regexp_replace(cleaned, r'\s+', ' ')
        
        # Trim
        cleaned = F.trim(cleaned)
        
        return df.withColumn(new_col, cleaned)
    
    def combine_text_columns(self, df, columns, new_col="combined_text"):
        """
        Combine plusieurs colonnes texte en une seule
        """
        print(f" Combinaison des colonnes: {columns}")
        
        # Vérifier que les colonnes existent
        existing_cols = [col for col in columns if col in df.columns]
        
        if len(existing_cols) != len(columns):
            missing = set(columns) - set(existing_cols)
            print(f"  Colonnes manquantes: {missing}")
        
        # Combiner avec espace
        combined = F.concat_ws(" ", *[F.col(col) for col in existing_cols])
        
        return df.withColumn(new_col, combined)
    
    def process_categories(self, df, categories_col="categories"):
        """
        Traite la colonne des catégories
        """
        print(f" Traitement des catégories")
        
        # Séparer les catégories en liste
        df = df.withColumn("category_list", F.split(F.col(categories_col), " "))
        
        # Nombre de catégories par article
        df = df.withColumn("num_categories", F.size(F.col("category_list")))
        
        # Catégorie principale (première)
        df = df.withColumn("main_category", F.col("category_list")[0])
        
        # Extraire le domaine (ex: 'cs' de 'cs.AI')
        df = df.withColumn(
            "domain",
            F.split(F.col("main_category"), "\\.")[0]
        )
        
        return df